# Shamugia et al. (2026) on dandiset 000630

Paper: Shamugia, Weiler, ... Goriounova and Mansvelder (2026), "Ion Channels and Morphology Drive
Widespread Resonance in Human Cortical Interneurons", Research Square preprint
[doi:10.21203/rs.3.rs-11136357/v1](https://doi.org/10.21203/rs.3.rs-11136357/v1). The claims file
(`claims/mansvelder-2026.yaml`) was written from version 1 of the preprint and its supplement.

Dataset: [DANDI:000630](https://dandiarchive.org/dandiset/000630/0.230915.2257), "Human L1
patch-seq electrophysiology" (Chartrand et al. 2023), version **0.230915.2257**. This is the
version the paper cites (reference 27, doi:10.48324/dandi.000630/0.230915.2257) and the only
published version; the draft has the same 210 assets and has not changed since the version was
published. The files are streamed; nothing is downloaded.

The dandiset holds the electrophysiology of human layer 1 interneurons recorded at the Allen
Institute, which is a part of the paper's human acute sample. It holds no mouse, cultured or
ZD7288 recordings, no transcriptomes, no morphologies and no cells outside layer 1. Subclass and
donor labels per cell come from the analysis repository of Chartrand et al. (2023), which the
dandiset's description names (github.com/AllenInstitute/patchseq_human_L1 at commit 7777857),
and are checked by the link claim `chartrand-label-link`.

| Claim | Tested here |
|---|---|
| chartrand-label-link (link claim added by this attempt) | tested |
| sample-size | tested (counts) |
| fig3eg-human-fres-sag-rin | tested on LAMP5 and VIP cells |
| fig3ik-human-q-sag-rin | tested on LAMP5 and VIP cells |
| figs1-region-pathology | tested |
| fig2b, fig2ab, fig2c, fig2d (three claims), figs2a, fig2e (two), fig2f, fig2g, fig3c, fig3d, fig3fh, fig3jl | blocked: need mouse recordings |
| fig2d-pv-highest | blocked: no PV or SST cells in the dandiset |
| fig4b, fig4c, fig4i, fig4df, fig4g, figs2-l23, fig4km | blocked: need cells across cortical depth, morphologies, or mouse cells |
| fig5be, figs3 | blocked: ZD7288 recordings not public |
| fig5g, fig5i, fig5jk, fig5l | blocked: cultured recordings not in the dandiset |
| fig6a, fig6b, fig6c, figs4 | blocked: transcriptomes not in the dandiset |
| fig7 and fig8 model claims (nine) | blocked: computational models, not attempted |

The resonance fractions per subclass are printed below as a check of the chirp analysis, but
the claims that compare them with mouse cells are not tested.

In [ ]:
import time, resource, warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
import resonance as R

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
print("dandiset", R.DANDISET, "version", R.VERSION)

## Loading

Every file is opened once; the chirp and hyperpolarizing long-square sweeps of cells that have a chirp are reduced as they are read (see `resonance.py`).

In [ ]:
t0 = time.time()
meta = R.version_metadata()
assets = R.list_assets()
files = R.process_files(assets, workers=8)
print(f"{len(files)} files read in {time.time() - t0:.0f} s; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print("version name:", meta["name"], "| assets in the version metadata:",
      meta["assetsSummary"]["numberOfFiles"], "| subjects:", meta["assetsSummary"]["numberOfSubjects"])

In [ ]:
rows = []
for r in files:
    d = dict(path=r["path"], subject=r["subject"]["subject_id"], session_id=r["session_id"],
             nwb_age=r["subject"].get("age"), nwb_sex=r["subject"].get("sex"),
             nwb_species=r["subject"].get("species"), has_chirp=r["cell"] is not None)
    c = r["cell"]
    if c is not None:
        d.update(c["drop"])
        d["n_chirp"] = len(c["chirps"])
        if c["chirps"]:
            for rd in "ab":
                f, amp, sm_ = R.impedance(c["chirps"], rd)
                d.update({f"{k}_{rd}": v for k, v in R.resonance(f, sm_).items()})
            d["vm_a"] = float(np.mean([x["vm"] for x in c["chirps"]]))
            d["chirp_start"] = c["chirps"][0]["start"]
            d["chirp_end"] = c["chirps"][0]["end"]
            d["chirp_i_peak"] = float(np.mean([x["i_peak"] for x in c["chirps"]]))
            d["chirp_v_pp"] = float(np.mean([np.ptp(x["v"]) for x in c["chirps"]]))
        d.update(R.sag_rin(c["ls"]))
        d["ls_amps"] = sorted({round(s["amp"]) for s in c["ls"]})
        d["ls_deflect_ok"] = int(sum(s["v_min"] < s["v_base"] for s in c["ls"]))
        d["ls_n"] = len(c["ls"])
    rows.append(d)
cells = pd.DataFrame(rows)

lab = R.chartrand_table()
keep = ["session_id", "spec_id", "donor_name", "subclass", "t-type", "structure",
        "medical_conditions", "donor_age", "donor_sex", "soma_depth_um", "layer_lims"]
cells = cells.merge(lab[keep], on="session_id", how="left")
chirp = cells[cells["n_chirp"].fillna(0) > 0].copy()
chirp["sub"] = chirp["subclass"].map({"LAMP5/PAX6/Other": "LAMP5", "VIP": "VIP"})
print(len(cells), "cells;", len(chirp), "with a usable 20 s chirp")

## Data check

In [ ]:
print("files:", len(cells), "| subjects:", cells["subject"].nunique(), "| sessions:", cells["session_id"].nunique())
print("species in the files:", cells["nwb_species"].value_counts().to_dict())
stim = {}
for r in files:
    for s in {x["stimulus"].rsplit("_DA", 1)[0] for x in r["sweeps"]}:
        stim[s] = stim.get(s, 0) + 1
print("files holding each chirp or long-square stimulus:",
      {k: v for k, v in sorted(stim.items()) if "CHIRP" in k or any(t in k for t in R.LS_NAMES)})
print("files with any chirp sweep:", int(cells["has_chirp"].sum()), "from", cells.loc[cells["has_chirp"], "subject"].nunique(), "subjects")
print("dropped chirp sweeps (not 20 s / stopped early / spiking):",
      int(cells["chirp_not20s"].sum()), "/", int(cells["chirp_truncated"].sum()), "/", int(cells["chirp_spike"].sum()))
print("files with chirp sweeps but none usable:", int((cells["has_chirp"] & (cells["n_chirp"] == 0)).sum()))
print("usable chirp sweeps per cell:", chirp["n_chirp"].value_counts().sort_index().to_dict())
print("chirp window start (s):", sorted(chirp["chirp_start"].round(2).unique()), "end (s):", sorted(chirp["chirp_end"].round(2).unique()))
print(f"chirp current amplitude (pA): {chirp['chirp_i_peak'].min():.1f} to {chirp['chirp_i_peak'].max():.1f}")
print(f"chirp voltage peak-to-peak after downsampling (mV): {chirp['chirp_v_pp'].min():.1f} to {chirp['chirp_v_pp'].max():.1f} "
      f"(median {chirp['chirp_v_pp'].median():.1f}; the Methods aim for +/- 5 mV)")
print(f"Vm before the chirp, reading a (mV): {chirp['vm_a'].min():.1f} to {chirp['vm_a'].max():.1f}, median {chirp['vm_a'].median():.1f} "
      "(the Methods bias cells to -65 mV)")
print(f"Vm before the long squares, reading b (mV): {chirp['vm_b'].min():.1f} to {chirp['vm_b'].max():.1f}")
print("hyperpolarizing long squares per chirp cell:", chirp["ls_n"].min(), "to", chirp["ls_n"].max(),
      "| dropped as stopped early:", int(chirp["ls_truncated"].sum()))
print("step amplitudes (pA) seen:", sorted({a for l in chirp["ls_amps"] for a in l}))
print("long squares whose minimum lies below the pre-step baseline:", int(chirp["ls_deflect_ok"].sum()), "of", int(chirp["ls_n"].sum()))
print("step used for sag reading b (pA):", chirp["step_b"].round(0).value_counts().to_dict())
for v, u in [("fres_a", "Hz"), ("Q_a", ""), ("f3db_a", "Hz"), ("fres_b", "Hz"), ("Q_b", ""), ("f3db_b", "Hz"),
             ("sag_a", ""), ("sag_b", ""), ("rin_a", "MOhm"), ("rin_b", "MOhm"), ("z0_a", "MOhm")]:
    print(f"  {v:7s} {chirp[v].min():9.3f} to {chirp[v].max():9.3f} {u}  (median {chirp[v].median():.3f}, NaN {chirp[v].isna().sum()})")
key = chirp[["z0_a", "zmax_a", "rin_a", "sag_a"]].round(6).astype(str).agg("|".join, axis=1)
print("chirp cells sharing identical impedance, sag and input resistance values with another cell (duplicate deposits):",
      int(key.duplicated(keep=False).sum()))
v0 = {r["session_id"]: tuple(np.round(r["cell"]["chirps"][0]["v"][:2000], 4)) for r in files if r["cell"] and r["cell"]["chirps"]}
print("chirp cells whose first second of chirp response equals another cell's:", len(v0) - len(set(v0.values())))

Units. Voltage is stored in mV with conversion 0.001 to volts, current in pA with conversion
1e-12 to amperes; the helper converts both, and the printed ranges (Vm near -64 mV, chirp
currents of a few to tens of pA, input resistances of hundreds of MOhm) agree with those units.
Sweeps carry their own stimulus, so there is no trial table to align: the check of timing is
that every hyperpolarizing step drives the voltage below its pre-step baseline inside the step
window read from the stimulus (count above). No two chirp cells share their measures or their
first second of chirp response, so no recording is deposited twice within the dandiset. The
subject fields are checked against the labels in the link claim.

## chartrand-label-link

Link claim (added by this attempt). The subclass and donor labels per cell come from the
Chartrand et al. (2023) table. Cells are matched by the session id of each NWB file, which is
the "Ephys_Roi_Result" id in the table's `nwb_path`. The link holds when every file matches one
row, each NWB subject id maps to one donor name and back, and sex and age agree between the files
and the table. Table S1 of the paper is used to say which of the dandiset's donors the paper
lists, and under which of its datasets.

In [ ]:
matched = cells["spec_id"].notna()
print("files matched to a row of the Chartrand table:", int(matched.sum()), "of", len(cells))
print("NWB subjects with more than one donor name:", int((cells.groupby("subject")["donor_name"].nunique() > 1).sum()),
      "| donor names with more than one NWB subject:", int((cells.groupby("donor_name")["subject"].nunique() > 1).sum()))
print("sex, files against table:")
print(pd.crosstab(cells["nwb_sex"], cells["donor_sex"]))
age_days = cells["nwb_age"].str.extract(r"P([\d.]+)D")[0].astype(float)
age_years = cells["donor_age"].astype(str).str.extract(r"(\d+)")[0].astype(float)
diff = (age_days / 365.25 - age_years)
print(f"age, files (days / 365.25) minus table (years): {diff.min():.2f} to {diff.max():.2f} over {diff.notna().sum()} files")
print("subclass of chirp cells (table):", chirp["subclass"].value_counts().to_dict())
print("t-type of chirp cells (table):", chirp["t-type"].value_counts().to_dict())
donors = sorted(cells["donor_name"].dropna().unique())
where = {d: [k for k, v in R.TABLE_S1_DONORS.items() if d in v] for d in donors}
print("dandiset donors in Table S1:", sum(bool(w) for w in where.values()), "of", len(donors))
print("Table S1 dataset of the dandiset donors:", pd.Series([",".join(w) or "absent" for w in where.values()]).value_counts().to_dict())
cd = sorted(chirp["donor_name"].unique())
print("Table S1 dataset of the donors of chirp cells:", pd.Series([",".join(where[d]) or "absent" for d in cd]).value_counts().to_dict())

## sample-size

> The dataset comprises 1085 human interneurons from 259 subjects (542 acute from 170 donors,
> 543 cultured from 89 donors) and 619 acutely recorded mouse interneurons from 293 mice.

The criterion counts the human acute cells and donors the attempt assembles, against 542 and
170, and says that on the dandiset alone this is not reproduced.

In [ ]:
print("human acute interneurons in the dandiset:", len(cells), "(expected 542 +/- 54.2)")
print("their donors:", cells["subject"].nunique(), "(expected 170 +/- 17)")
print("cells with a usable 20 s chirp:", len(chirp), "from", chirp["subject"].nunique(), "donors (paper: 297 human acute cells with a chirp)")
print("of those, by subclass:", chirp["sub"].value_counts().to_dict(), "(paper Table S3: LAMP5 144, VIP 60, PV 78, SST 15)")
print("donors of the dandiset listed in Table S1 as 'Chartrand':",
      sum("Chartrand" in where[d] for d in donors), "of", len(TABLE_S1 := R.TABLE_S1_DONORS["Chartrand"]), "Table S1 'Chartrand' donors")
print("Table S1 donors by dataset (rows read from the supplement):", {k: len(v) for k, v in R.TABLE_S1_DONORS.items()})

**Result: not reproduced on the dandiset alone, as the criterion foresees.** The dandiset holds 210 human acute interneurons from 53 donors, against 542 from 170; 85 of them have a usable chirp. Every one of its 53 donors is in Table S1, 43 of them under the dataset "AcuteLDPublished" and 10 under "Chartrand". No mouse or cultured cells are in it.

## Chirp analysis and its readings

fres, Q and f3dB per cell under the two readings of O1 (frequencies kept from 0.2 Hz, IPFX's
default, or from 0.4 Hz, the stimulus band). The resonant fraction per subclass is set against the
human fractions of Table S3 (LAMP5 45 of 144 = 31.25%, VIP 15 of 60 = 25.0%) as a check of which
reading follows the paper's computation; the claims that compare these fractions with mouse cells
are blocked.

In [ ]:
for rd in "ab":
    res = R.is_resonant(chirp[f"fres_{rd}"], chirp[f"Q_{rd}"])
    by = chirp.assign(r=res).groupby("sub")["r"].agg(["sum", "count", "mean"])
    print(f"reading {rd}: resonant {int(res.sum())} of {len(res)} ({100 * res.mean():.1f}%);",
          {k: f"{int(v['sum'])}/{int(v['count'])} = {100 * v['mean']:.1f}%" for k, v in by.iterrows()})
    print("   medians by subclass:", chirp.groupby("sub")[[f"fres_{rd}", f"Q_{rd}", f"f3db_{rd}"]].median().round(3).to_dict("index"))
    print(f"   cells whose peak is the lowest frequency kept (Q = 1): {(chirp[f'Q_{rd}'] == 1).sum()}; "
          f"cells with no 3 dB point above the peak: {chirp[f'f3db_{rd}'].isna().sum()}")
for q in (1.1, 1.2):
    print(f"fres > 1 Hz and Q > {q}, reading a: {100 * R.is_resonant(chirp['fres_a'], chirp['Q_a'], q).mean():.1f}%")
print("sag and input resistance medians by subclass:", chirp.groupby("sub")[["sag_a", "sag_b", "rin_a", "rin_b"]].median().round(3).to_dict("index"))
print("Spearman rho between readings: fres", round(stats.spearmanr(chirp.fres_a, chirp.fres_b)[0], 3),
      "Q", round(stats.spearmanr(chirp.Q_a, chirp.Q_b)[0], 3), "sag", round(stats.spearmanr(chirp.sag_a, chirp.sag_b)[0], 3),
      "rin", round(stats.spearmanr(chirp.rin_a, chirp.rin_b)[0], 3))

## Mixed models used by the claims of Figure 3

Linear mixed models with a random intercept for donor, fitted by REML in statsmodels (O5,
reading b), with Wald tests on each coefficient. With two subclasses each interaction has one
degree of freedom, so its Wald test is the test of the term. Reading (a) of O5 (Satterthwaite F
tests, as SPSS MIXED or lmerTest) is not available in this Python environment; GEE with an
exchangeable correlation within donor is shown as a variant, as the paper confirmed its models
with GEE.

In [ ]:
def fit(formula, data, gee=False):
    if gee:
        return smf.gee(formula, groups=data["subject"], data=data, cov_struct=sm.cov_struct.Exchangeable()).fit()
    return smf.mixedlm(formula, data, groups=data["subject"]).fit(reml=True)

def terms(f, keys):
    return {k: (round(float(f.params[k]), 5), float(f"{f.pvalues[k]:.3g}")) for k in keys}

def frame(y, rd, sr):
    return chirp.rename(columns={f"{y}_{rd}": "y", f"sag_{sr}": "sag", f"rin_{sr}": "rin"})[["y", "sag", "rin", "sub", "subject"]].dropna()

print("cells per donor among chirp cells:", chirp.groupby("subject").size().describe()[["count", "min", "50%", "max"]].to_dict())

## fig3eg-human-fres-sag-rin

> In human interneurons, sag ratio and input resistance predict peak resonance frequency in a
> subclass-specific way; fres rises with sag ratio in LAMP5 (beta 3.31), VIP (4.79) and PV (6.13)
> cells and falls with input resistance in LAMP5 (-0.002) and PV (-0.006) cells.

Model of fres on sag, input resistance, subclass and both interactions with subclass, donor
intercept, then per subclass fres on sag and input resistance (criterion). The dandiset has no PV
or SST cells, so the PV slopes cannot be tested and the interactions compare LAMP5 with VIP only.
The main reading is (a) for O1, O2 and O3 (see the summary for why); every combination is shown.

In [ ]:
out = []
for rd in "ab":
    for sr in "ab":
        d = frame("fres", rd, sr)
        for gee in (False, True):
            f = fit("y ~ sag * C(sub) + rin * C(sub)", d, gee)
            row = dict(chirp=rd, sag_rin=sr, model="GEE" if gee else "LMM", n=len(d),
                       converged=getattr(f, "converged", None),
                       **{f"{k} (b, p)": v for k, v in terms(f, ["sag:C(sub)[T.VIP]", "rin:C(sub)[T.VIP]"]).items()})
            for s, g in d.groupby("sub"):
                fs = fit("y ~ sag + rin", g, gee)
                for k, v in terms(fs, ["sag", "rin"]).items():
                    row[f"{s} {k} (b, p)"] = v
            out.append(row)
fig3eg = pd.DataFrame(out)
print(fig3eg.to_string())
main = fig3eg.query("chirp == 'a' and sag_rin == 'a' and model == 'LMM'").iloc[0]
print("\nmain reading: LAMP5 sag", main["LAMP5 sag (b, p)"], "(expected 3.31 +/- 0.331);",
      "VIP sag", main["VIP sag (b, p)"], "(expected 4.79 +/- 0.479);",
      "LAMP5 rin", main["LAMP5 rin (b, p)"], "(expected -0.002 +/- 0.0002);",
      "VIP rin", main["VIP rin (b, p)"], "(paper p = 0.088)")
print("LAMP5 cells", (chirp["sub"] == "LAMP5").sum(), "from", chirp.loc[chirp["sub"] == "LAMP5", "subject"].nunique(),
      "donors; VIP cells", (chirp["sub"] == "VIP").sum(), "from", chirp.loc[chirp["sub"] == "VIP", "subject"].nunique(), "donors")

**Result: partially reproduced, under the assumed readings (a).** fres rises with sag in LAMP5 (b = 3.70, p = 0.0004) and VIP cells (b = 4.40, p = 0.024) and falls with input resistance in LAMP5 cells (b = -0.0037, p = 0.023), the signs and significance the paper reports; of the three slopes only the VIP sag slope is within tolerance (LAMP5 sag 3.70 against 3.31 +/- 0.33, LAMP5 input resistance -0.0037 against -0.002 +/- 0.0002). The interactions with subclass are not significant (sag p = 0.82, input resistance p = 0.33), but with LAMP5 and VIP only they are not the paper's four-subclass test. The PV slopes cannot be tested. The signs hold in every reading and in GEE; the VIP sag slope loses significance under sag and input resistance reading (b), and the LAMP5 input resistance slope under chirp reading (b) with sag reading (a) (p = 0.058). GEE gave no estimate for the VIP model (NaN).

## fig3ik-human-q-sag-rin

> In human interneurons, sag ratio strongly predicts resonance strength (beta 0.47, p < 0.001)
> with no subclass difference, and input resistance is negatively related to it (beta -0.0001,
> p = 0.005).

Model of Q as for fig3eg. The slopes are from a model without the interactions, the full model's
shown as a variant (criterion).

In [ ]:
out = []
for rd in "ab":
    for sr in "ab":
        d = frame("Q", rd, sr)
        for gee in (False, True):
            full = fit("y ~ sag * C(sub) + rin * C(sub)", d, gee)
            red = fit("y ~ sag + rin + C(sub)", d, gee)
            out.append(dict(chirp=rd, sag_rin=sr, model="GEE" if gee else "LMM", n=len(d),
                            converged_full=getattr(full, "converged", None), converged_reduced=getattr(red, "converged", None),
                            **{f"reduced {k} (b, p)": v for k, v in terms(red, ["sag", "rin"]).items()},
                            **{f"full {k} (b, p)": v for k, v in terms(full, ["sag", "rin", "sag:C(sub)[T.VIP]"]).items()}))
fig3ik = pd.DataFrame(out)
print(fig3ik.to_string())
m = fig3ik.query("chirp == 'a' and sag_rin == 'a' and model == 'LMM'").iloc[0]
print("\nmain reading: sag", m["reduced sag (b, p)"], "(expected 0.47 +/- 0.047, p < 0.001);",
      "rin", m["reduced rin (b, p)"], "(expected -0.0001 +/- 0.00001, p < 0.05);",
      "sag by subclass", m["full sag:C(sub)[T.VIP] (b, p)"], "(expected p >= 0.05)")

**Result: not reproduced, under the assumed readings (a).** Q rises with sag (b = 0.150, p = 0.0046) without a subclass difference (p = 0.43), but the effect is about a third of the paper's 0.47 and does not reach p < 0.001, and input resistance is not related to Q (b = -0.00005, p = 0.24). Under chirp reading (b) with sag and input resistance reading (a) the three tests hold (sag p = 3e-05, input resistance p = 0.047), which would make it partially reproduced; under the other readings it is not. In no reading is the sag slope near 0.47. The mixed model without interactions did not converge under the main reading; GEE on the same terms gives sag b = 0.138 (p = 0.0006) and input resistance p = 0.11, so the input resistance part fails there too.

## figs1-region-pathology

> Resonance metrics (fres, Q, 3 dB cutoff) and input resistance do not differ between brain
> regions of resection or between epilepsy and tumour cases; sag ratio does.

Lobe and disease per donor from Table S1 (criterion). The lobe is Table S1's "Lobe" column where
it is filled, and otherwise its "Area" grouped into lobes (MTG, STG, ITG, TemL temporal; SFG,
MFG, IFGtr, PRG, LOrG, FroL frontal; SPL, AnG, ParL parietal; IOG, SOG, OccL occipital).
Disease is Table S1's "Disease category" (epilepsy or epileptiform against tumour; other
categories left out). The same labels from the Chartrand table (cell structure, donor medical
condition) are shown as a variant. Kruskal-Wallis across lobes and two-sided Mann-Whitney U
between epilepsy and tumour (Fig. S1 legend). Only acute cells exist here, so readings (a) and
(b) of the criterion are the same.

In [ ]:
area_lobe = {"MTG": "temporal", "STG": "temporal", "ITG": "temporal", "TemL": "temporal", "SFG": "frontal",
             "MFG": "frontal", "IFGtr": "frontal", "PRG": "frontal", "LOrG": "frontal", "FroL": "frontal",
             "SPL": "parietal", "AnG": "parietal", "ParL": "parietal", "IOG": "occipital", "SOG": "occipital", "OccL": "occipital"}
lobe_col = {"TemL": "temporal", "FroL": "frontal", "ParL": "parietal", "OccL": "occipital"}
def s1_lobe(d):
    ds, area, dis, lobe = R.TABLE_S1.get(d, ("", "", "", ""))
    return lobe_col.get(lobe) or area_lobe.get(area)
def disease(x):
    x = str(x).lower()
    return "tumour" if "tumor" in x else ("epilepsy" if "epilep" in x else None)
chirp["lobe"] = chirp["donor_name"].map(s1_lobe)
chirp["disease"] = chirp["donor_name"].map(lambda d: disease(R.TABLE_S1.get(d, ("", "", "", ""))[2]))
chirp["lobe_ct"] = chirp["structure"].map(area_lobe)
chirp["disease_ct"] = chirp["medical_conditions"].map(lambda x: disease(x) if isinstance(x, str) else None)
print("lobes (Table S1):", chirp["lobe"].value_counts(dropna=False).to_dict(), "| donors:", chirp.groupby("lobe")["subject"].nunique().to_dict())
print("disease (Table S1):", chirp["disease"].value_counts(dropna=False).to_dict(), "| donors:", chirp.groupby("disease")["subject"].nunique().to_dict())
print("Table S1 against Chartrand table, lobe agreement:", int((chirp["lobe"] == chirp["lobe_ct"]).sum()), "of",
      int((chirp["lobe"].notna() & chirp["lobe_ct"].notna()).sum()), "; disease agreement:",
      int((chirp["disease"] == chirp["disease_ct"]).sum()), "of", int((chirp["disease"].notna() & chirp["disease_ct"].notna()).sum()))

def region_tests(lcol, dcol):
    rows = []
    for v in ["fres_a", "Q_a", "f3db_a", "rin_a", "sag_a", "fres_b", "Q_b", "f3db_b", "rin_b", "sag_b"]:
        d = chirp.dropna(subset=[lcol])
        kw = stats.kruskal(*[g[v].values for _, g in d.groupby(lcol)])
        e = chirp.loc[chirp[dcol] == "epilepsy", v]; t = chirp.loc[chirp[dcol] == "tumour", v]
        mw = stats.mannwhitneyu(e, t, alternative="two-sided")
        rows.append(dict(measure=v, KW=round(kw.statistic, 2), KW_p=round(kw.pvalue, 3), MWU=mw.statistic,
                         MWU_p=round(mw.pvalue, 4), n_epi=len(e), n_tum=len(t), med_epilepsy=round(e.median(), 3),
                         med_tumour=round(t.median(), 3), **{f"med_{l}": round(g[v].median(), 3) for l, g in d.groupby(lcol)}))
    return pd.DataFrame(rows)
print("\nTable S1 labels (main):")
print(region_tests("lobe", "disease").to_string())
print("\nChartrand table labels (variant):")
print(region_tests("lobe_ct", "disease_ct").to_string())

**Result: reproduced, under the assumed reading (a).** With Table S1's labels, fres, Q, f3dB and input resistance differ neither between lobes (Kruskal-Wallis p = 0.37, 0.65, 0.76 and 0.23) nor between epilepsy and tumour cases (Mann-Whitney p = 0.097, 0.35, 1.00 and 0.84), and none reaches p < 0.05 under the chirp reading (b) or with the Chartrand table's labels. Sag ratio differs between lobes (p = 0.044; the paper p < 0.001) but not between epilepsy and tumour (p = 0.94, medians 0.131 and 0.125), where the paper finds it lower in epilepsy (p < 0.001); with the Chartrand table's labels, which cover 47 of the 85 cells, sag is lower in epilepsy (p = 0.014). The sample is small: 74 temporal, 8 frontal and 1 parietal cell, and 18 tumour cells from 6 donors.**

## Summary

| Claim | Outcome |
|---|---|
| chartrand-label-link | reproduced |
| sample-size | not reproduced (the dandiset holds part of the sample) |
| fig3eg-human-fres-sag-rin | partially reproduced (assumed readings) |
| fig3ik-human-q-sag-rin | not reproduced (assumed readings) |
| figs1-region-pathology | reproduced (assumed readings) |
| the other 40 claims | blocked (see the attempt record) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The dandiset holds 85 human layer 1 cells with a usable
chirp (LAMP5/PAX6/Other 67, VIP 18) where the paper's human acute sample has 297, with PV and SST
cells; the tested claims are run on these. The paper's chirp, sag and input resistance code is not
public, so the IPFX v2.0.0 computations are used (readings a of O1 to O3) and the alternatives are
shown. Reading (a) of O1 is adopted because the Methods say the recordings were analysed with
IPFX and because its resonant fractions (LAMP5 32.8%, VIP 22.2%) are near Table S3's human
fractions (31.25% and 25.0%) while reading (b)'s are not (20.9% and 16.7%). The mixed models use
Wald tests in statsmodels in place of Satterthwaite F tests.

What was found about the dataset. Every file links to the Chartrand et al. table by its session
id; sex and age agree. Of the 210 files, 93 hold chirp sweeps and 85 have at least one usable
20 s chirp; many sweeps stop early (their voltage is zero-filled), which the files do not mark.
All 53 donors are in the paper's Table S1, 43 of them under the dataset "AcuteLDPublished" (Lee,
Dalley et al. 2023) and 10 under "Chartrand". The files carry no subclass, layer, structure,
disease or donor name; these come from the Chartrand repository and Table S1. Table S1 and the
Chartrand table agree on the lobe of 81 of 83 cells and the disease of 45 of 47.